# GCI Global 2026 — Home Credit Default Risk (`v2` Submitted Pipeline)

**Submitted Solution:** `v2` (`submissions_v2/sub_final_v2.csv`, `ENABLE_LINKAGE = False`) — an equal-weight fold-isolated rank blend of **20 models across 10 model-view groups** (`LightGBM`, `XGBoost`, and `CatBoost` over curated views `VIEW_A`, `VIEW_B`, `VIEW_C`), validated with 5-fold and 10-fold `StratifiedGroupKFold` on repeat-applicant fingerprint `K3`.

- **Validation AUC (`v2 --full`):** **`0.766841`** mean 5-fold group-CV AUC | **`0.766777`** pooled OOF AUC (`+0.001662` across `5/5` folds vs `v1` on `folds_group.npy`).
- **Fast Verification (`--fast`):** 4 LightGBM models (`VIEW_A` x3 seeds + `VIEW_B` seed 42), `~20–27 min`, **`0.765773`** pooled OOF AUC.
- **Full Reproduction (`--full`):** All 20 models from scratch, `~4.2 hours` (`14,983.5s` uncached on 2 physical / 4 logical CPU cores).
- **History Note:** `v1` and grey-zone linkage (`ENABLE_LINKAGE = True`) files are kept only as historical records.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "run_all_v2.py").exists():
    raise FileNotFoundError("Run this notebook from the repository root containing src/run_all_v2.py")

os.environ["PYTHONHASHSEED"] = "42"

required_inputs = ["train.csv", "test.csv", "sample_submission.csv"]
missing = [f for f in required_inputs if not (ROOT / "input" / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing raw data in input/: {missing}. See input/README.txt.")

print("Python:", sys.version.split()[0])
print("Repo root:", ROOT)
print("Input files verified:", required_inputs)

In [ ]:
# Run deterministic fast verification (~20-27 min):
# Equivalent CLI: python src/run_all_v2.py --fast
# Full 20-model reproduction (~4.2 h uncached):
# subprocess.run([sys.executable, "src/run_all_v2.py", "--full", "--fresh"], env={**os.environ, "PYTHONHASHSEED": "42"}, check=True)

FAST_CSV = ROOT / "submissions_v2" / "sub_fast_v2.csv"
FAST_MANIFEST = ROOT / "experiments_v2" / "manifest_fast.json"

cmd = [
    sys.executable,
    "src/run_all_v2.py",
    "--fast",
    "--fresh",
    "--out",
    str(FAST_CSV),
    "--manifest",
    str(FAST_MANIFEST),
]
subprocess.run(cmd, env={**os.environ, "PYTHONHASHSEED": "42"}, check=True)

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = json.loads((ROOT / "experiments_v2" / "manifest.json").read_text(encoding="utf-8"))

expected_fast = manifest.get("expected_fast_sha256")
if FAST_CSV.exists():
    fast_sha = sha256_file(FAST_CSV)
    print(f"Produced --fast CSV : {FAST_CSV}")
    print(f"Observed SHA256     : {fast_sha}")
    print(f"Expected SHA256     : {expected_fast}")
    assert fast_sha == expected_fast, f"Fast SHA256 mismatch: {fast_sha} != {expected_fast}"
    print("--fast SHA256 MATCH VERIFIED")

full_csv = ROOT / "submissions_v2" / "sub_final_v2.csv"
expected_full = manifest.get("submission_sha256")
if full_csv.exists() and expected_full:
    full_sha = sha256_file(full_csv)
    print(f"Shipped --full CSV  : {full_csv}")
    print(f"Observed SHA256     : {full_sha}")
    print(f"Expected SHA256     : {expected_full}")
    assert full_sha == expected_full, f"Full SHA256 mismatch: {full_sha} != {expected_full}"
    print("--full SHA256 MATCH VERIFIED")